I have spent three weeks measuring how this game is actually played:
[reading the top's DNA](https://www.kaggle.com/code/destbreso/a-dna-test-for-agents),
[X-raying the week-four board](https://www.kaggle.com/code/destbreso/a-week-four-x-ray-of-the-top-twelve),
[watching everyone play the same opening](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening).
Every one of those notebooks studied schedules that other people wrote.
This one is about the other direction: instead of copying a base schedule
or deducing one from replays, SEARCH for your own. I am publishing the
reasoning, the algorithms (a schedule compiler and an island genetic
algorithm on top of it), and the engine findings the build surfaced. I am
not publishing the schedule the search returns, and section 8 says exactly
why, because the argument for keeping it is itself the thesis.

Everything else is runnable code: the companion repo
[kaggriculture-island-ga](https://github.com/destbreso/kaggriculture-island-ga)
carries the versioned implementation under MIT (the compiler, a reference
executor, the island GA, a CLI that ends in screen/confirm metrics). This
notebook keeps the reasoning and the pseudocode; the repo keeps the code
you can run with your own parameters.

**TL;DR, with links into the sections:**

1. [The winning shape on this ladder is a base schedule plus a thin
   adaptive layer](#s1), and the evidence is public.
2. [Four roads lead to a base schedule](#s2): hand rules, RL, copying,
   and search. Search is the underpriced one.
3. [Price the corridor before searching](#s3): an engine-exact relaxed
   bound puts a crude roof near \\$229k against an idle opponent, more
   than three times what the public consensus route banks in the same
   setting. The gap is the prize.
4. [The bound's optimal mix already includes tomato and egg at MEDIAN
   demand](#s3): the rebalanced goods belong in a schedule before any
   scarcity spike.
5. [Two legal representations, one law](#s4): the financing chain must
   survive every mutation. A spec+compiler genome re-derives the market
   channel from scratch; a deltas-over-stream genome edits one recorded
   stream in place and keeps the choreography, which MEASURES as 74 %
   of a route's value. Splicing two streams is the one illegal move.
6. [The compiler is the real algorithm](#s5): five subsystems derived
   from twelve numbers, plus four engine asymmetries that punish any
   cautious scheduler.
7. [Islands beat a single population here for reasons you can measure](#s6),
   and seeding them with known schedules would be a mistake for a
   non-obvious reason: [selection takeover plus transcription loss](#s61).
8. [The campaign so far](#s7): what the searches measured that nobody
   guessed: gains ALONE and gains IN COMPANY are different axes (one
   search won +13.2k alone and lost 13.9k accompanied on the same
   deltas), so the current objective is priced in LADDER currency,
   Pr[win] = Phi(margin/sigma), over wide simulator panels.
   [What stays home](#s8).

<a id="s1"></a>
## 1. The winning shape: a base plus a thin layer

Whole-ladder texture measurement says this field plays SCRIPTS. In the
[week-four X-ray](https://www.kaggle.com/code/destbreso/a-week-four-x-ray-of-the-top-twelve)
I classified every turn of ~2,000 scored submissions against that
submission's own modal plan: above a derived border near score 950 the
ladder is one scripted continent, median plan-deviation 0.2 % at every
altitude, and the deviations that do exist concentrate in the market
channel and the late game. Luka Duvanov reached the same conclusion from
a different angle in
[The top agent is a 720-turn replay, not a strategy](https://www.kaggle.com/code/nekkon/the-top-agent-is-a-720-turn-replay-not-a-strategy).

The nuance that matters: the very top carries a LIVE minority. A handful
of summit submissions re-plan whole episodes, and in
[Mutants at the top](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay)
the best agent on the board was the least plan-fixed of all. So live play
wins, but it is rare and expensive to build, and even the live agents
visibly START from a base: their episodes agree turn-for-turn with a
fixed spine early and diverge late. The economic reading is simple. Your
base schedule sets your floor; your adaptive layer buys points on top of
it; and the two investments are separable. This notebook is about buying
a better floor.

<a id="s2"></a>
## 2. Four roads to a base schedule, and what each costs

**Hand-written rules.** A rules agent must solve routing, financing and
crop timing at once, and every subsystem interacts with the others
through labour and cash. My own attempts at this road produced agents
that lose to mid-table scripts. Rules CAN reach the summit, the live
minority proves it, but that road's cost is the highest on the board.

**Reinforcement learning.** 720 decisions per episode, a combinatorial
action space per turn (several units, each with a verb and a target,
plus a market list), reward only at the end, and an opponent who shares
your market. Nothing here is impossible, but the sample cost is brutal
for a competition with five submissions a day, and the public evidence
is that nobody at the top is doing it.

**Copying.** Replay a strong team's recorded actions. It works, it is
how a large fraction of this ladder exists (the
[opening-book measurement](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening)
found 94 % of seats on a shared schedule at turn 24), and its ceiling is
exactly the agent you copied, minus everything that made it adaptive. It
also concentrates the meta: when half the board plays the same schedule,
they share one market and crowd each other's prices.

**Search.** The game engine is public, deterministic given a seed, and
fast (a full episode against an idle opponent runs in seconds). A season
is a fixed 30-day horizon. That makes the base schedule an OFFLINE
optimisation problem: I framed the class in
[what kind of optimisation is this](https://www.kaggle.com/code/destbreso/kaggriculture-what-kind-of-optimisation-is-this),
and scheduling problems of this shape are NP-hard in general [5], which
licenses heuristics rather than exact solvers.

The moat argument, stated honestly. A schedule found by search is unique
while nobody else fields it, and on a shared market uniqueness is worth
real money: an uncrowded crop mix sells into better prices. Yes, the
moment I field it, anyone can deduce most of it from replays; I published
[the tool that does exactly that](https://www.kaggle.com/code/destbreso/a-dna-test-for-agents)
and [the dataset to do it with](https://www.kaggle.com/datasets/destbreso/kaggriculture-top10-replay-streams).
So the durable advantage is not any single schedule. It is the
INFRASTRUCTURE that grows them: whoever can regrow a competitive base
faster than the field copies it keeps the edge. That is why publishing
the method costs me less than it appears, and why the result itself
stays home.

<a id="s3"></a>
## 3. Price the corridor before searching

Before spending compute on a search, bound the prize. The engine's price
function is public (I verified the 1.32.7 parameter set against live
prices in
[this thread](https://www.kaggle.com/competitions/kaggriculture/discussion/734412)),
so the end-of-season revenue of selling S units of a good into median
demand is an exact sum, and a RELAXED optimum (ignore walking, watering,
action budgets, financing timing: everything ignored only lowers the
truth) is a greedy allocation of tile-days to crops by marginal revenue,
with animal products at their engine caps and a \\$2 marginal trim. The
pricing code below is the engine's own `market_price`, inlined verbatim
with its constants, so the bound is reproducible from this cell alone.

In [1]:
import math

# engine constants and pricing, verbatim from kaggriculture.py (1.32.7)
MARKET_I0, HINGE_GAIN, PRICE_FLOOR = 10000, 8.0, 1
MARKET_PARAMS = {
 "WHEAT":      {"base": 25,  "T": 400, "below_func": "sqrt",   "below_target": 0.80, "above_func": "log",    "above_target": 0.20},
 "CARROT":     {"base": 35,  "T": 450, "below_func": "hinge",  "below_target": 1.00, "above_func": "sqrt",   "above_target": 0.70},
 "TOMATO":     {"base": 60,  "T": 200, "below_func": "hinge",  "below_target": 0.40, "above_func": "sqrt",   "above_target": 0.60},
 "STRAWBERRY": {"base": 120, "T": 100, "below_func": "sqrt",   "below_target": 0.70, "above_func": "linear", "above_target": 1.60},
 "MELON":      {"base": 250, "T": 300, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.60},
 "EGG":        {"base": 50,  "T": 332, "below_func": "hinge",  "below_target": 0.40, "above_func": "log",    "above_target": 0.20},
 "MILK":       {"base": 160, "T": 122, "below_func": "sqrt",   "below_target": 0.60, "above_func": "linear", "above_target": 1.60},
 "WOOL":       {"base": 200, "T": 105, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.20},
 "FERTILIZER": {"base": 100, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40}}

def _shape(func, x, T=None):
    x = max(0.0, x)
    if func == "linear": return x
    if func == "sq":     return x * x
    if func == "sqrt":   return math.sqrt(x)
    if func == "log":    return math.log(1.0 + x)
    if func == "log10":  return math.log10(1.0 + x)
    if func == "hinge":
        if not T or T <= 0:
            return x
        u = x / T
        return u + HINGE_GAIN * max(0.0, u - 1.0) ** 2
    return x

def price(item, inventory):
    p = MARKET_PARAMS[item]
    base, T = p["base"], p["T"]
    if inventory < MARKET_I0:
        f = p["below_func"]
        amp = p["below_target"] * base / _shape(f, T, T)
        val = base + amp * _shape(f, MARKET_I0 - inventory, T)
    else:
        f = p["above_func"]
        amp = p["above_target"] * base / _shape(f, T, T)
        val = base - amp * _shape(f, inventory - MARKET_I0, T)
    return max(PRICE_FLOOR, int(round(val)))

# median season demand draw (community-measured shop consumption)
DEMAND = {"WHEAT": 504, "STRAWBERRY": 396, "MILK": 288, "CARROT": 270,
          "TOMATO": 180, "WOOL": 180, "EGG": 180, "MELON": 30, "FERTILIZER": 0}
CROP = {"WHEAT": (2, 4, 6, False), "CARROT": (2, 3, 4, False),
        "TOMATO": (8, 8, 4, True), "STRAWBERRY": (10, 10, 4, True),
        "MELON": (10, 12, 6, False)}
TILE_DAYS = 60 * 28                     # ~60 workable tiles x 28 useful days
ANIM_CAP = {"EGG": 4 * 27, "MILK": 6 * 12, "WOOL": 6 * 9}
FERT_UNITS = 16 * 29                    # nightly fertilizer at full herd
COSTS = 60 * 10 + 5 * 400 + 6 * 500 + 4 * 300 + 1000 + 2000 + 300 * 20

def R(g, D, S):                         # exact end-of-season sale integral
    return sum(price(g, MARKET_I0 - D + i) for i in range(int(S)))

def per_tile_day(c):
    fy, my, y, ongoing = CROP[c]
    return (y / my) if not ongoing else (y + (30 - fy)) / 30

alloc = {c: 0 for c in CROP}
units = {g: 0.0 for g in DEMAND}
used, step = 0, 14
while used + step <= TILE_DAYS:
    best, best_gain, best_add = None, 1.0, 0
    for c in CROP:
        add = per_tile_day(c) * step
        gain = R(c, DEMAND[c], units[c] + add) - R(c, DEMAND[c], units[c])
        if gain > best_gain:
            best, best_gain, best_add = c, gain, add
    if not best:
        break
    units[best] += best_add
    alloc[best] += step
    used += step

rows, gross = [], 0
for g in DEMAND:
    S = units.get(g, 0.0)
    if g in ANIM_CAP:
        S = ANIM_CAP[g]
    if g == "FERTILIZER":
        S = FERT_UNITS
    S = int(S)
    while S > 0 and price(g, MARKET_I0 - DEMAND[g] + S - 1) <= 2:
        S -= 1
    r = R(g, DEMAND[g], S)
    rows.append((g, S, int(r)))
    gross += r
rows.sort(key=lambda t: -t[2])
print(f"{'good':11s} {'units':>6s} {'revenue':>9s}")
for g, S, r in rows:
    print(f"{g:11s} {S:6d} {r:9,d}")
BOUND = int(3000 + gross - COSTS)
print(f"\nCRUDE RELAXED BOUND vs idle opponent: ${BOUND:,}")

In [2]:
import matplotlib.pyplot as plt

HINGE = {"TOMATO": "#b5486b", "CARROT": "#c2571f", "EGG": "#b3872a"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

xs = list(range(0, 601, 5))
for g, col in HINGE.items():
    ax1.plot(xs, [price(g, MARKET_I0 - x) for x in xs], color=col, lw=2, label=g)
    T = MARKET_PARAMS[g]["T"]
    if T <= 600:
        ax1.plot([T], [price(g, MARKET_I0 - T)], "o", color=col, ms=6)
for g in ("WHEAT", "MELON"):
    ax1.plot(xs, [price(g, MARKET_I0 - x) for x in xs], color="#9aa3ad",
             lw=1.4, ls="--")
    ax1.annotate(g.lower(), (xs[-1], price(g, MARKET_I0 - 600)),
                 xytext=(-2, 5), textcoords="offset points", ha="right",
                 fontsize=8, color="#9aa3ad")
ax1.legend(frameon=False, fontsize=9, loc="upper left")
ax1.set_xlabel("units short of the reference inventory")
ax1.set_ylabel("unit price, $")
ax1.set_title("the 1.32.7 scarcity branch: the hinge trio\nturns convex at its knee (dots)", fontsize=10)
for s in ("top", "right"):
    ax1.spines[s].set_visible(False)

names = [g for g, _S, _r in rows if _r > 0][::-1]
revs = [r / 1000 for _g, _S, r in rows if r > 0][::-1]
cols = [HINGE.get(g, "#2f6fb2") for g in names]
b = ax2.barh(names, revs, color=cols, height=0.6)
for r_, v in zip(b, revs):
    ax2.annotate(f"{v:,.1f}k", (v, r_.get_y() + r_.get_height() / 2),
                 xytext=(4, 0), textcoords="offset points", va="center",
                 fontsize=8, color="#333")
ax2.set_xlabel("revenue in the relaxed optimum, thousands of $")
ax2.set_title("the bound's mix: strawberry leads, and the\nhinge trio (colored) enters at MEDIAN demand", fontsize=10)
ax2.set_xlim(0, max(revs) * 1.18)
for s in ("top", "right"):
    ax2.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [3]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.set_xlim(-8, 252)
ax.set_ylim(-0.9, 1.0)
ax.axhline(0, color="#d9dde3", lw=1.2, zorder=1)

ax.axvspan(84, 102, color="#7b5bb5", alpha=0.14, lw=0)
ax.axvspan(147, 164, color="#1f9d6b", alpha=0.14, lw=0)
ax.plot([72.8], [0], "o", ms=11, color="#2f6fb2", zorder=3)
ax.plot([92], [0], "o", ms=11, color="#7b5bb5", zorder=3)
ax.plot([155], [0], "o", ms=11, color="#1f9d6b", zorder=3)
ax.plot([229.5], [0], "o", ms=11, color="#c2571f", zorder=3)

ax.annotate("public consensus route\nreplayed vs idle: 72.8k\n(3-seed mean)",
            (72.8, 0), xytext=(0, -58), textcoords="offset points",
            ha="center", va="bottom", fontsize=9, color="#2f6fb2")
ax.annotate("week-four top twelve\nmedian banks: 92k, best 102k",
            (92, 0), xytext=(0, 16), textcoords="offset points",
            ha="center", fontsize=9, color="#5b3fa0")
ax.annotate("the public frontier: naked recorded\nstreams 147-154k, routed\nagents to 164k (vs idle)",
            (155, 0), xytext=(0, 14), textcoords="offset points",
            ha="center", fontsize=9, color="#14724d")
ax.annotate("crude relaxed bound: 229k\n(logistics ignored)",
            (229.5, 0), xytext=(0, -44), textcoords="offset points",
            ha="center", va="bottom", fontsize=9, color="#c2571f")
ax.annotate("", xy=(226, 0.52), xytext=(168, 0.52),
            arrowprops=dict(arrowstyle="<->", color="#666", lw=1.2))
ax.annotate("what even the frontier\nleaves unclaimed", (197, 0.62),
            ha="center", fontsize=9, style="italic", color="#666")
ax.set_yticks([])
ax.set_xlabel("bank at end of season, thousands of $")
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

Three readings of that table and scale.

**The corridor is wide.** The public consensus construction, executed
faithfully against an idle opponent, banks 72.8k averaged over three
seeds. The week-four top twelve bank 91.8k median per team in real
games, the best of them just above 100k. The relaxed roof sits at
\\$229,450. Even discounting the roof heavily for everything it ignores,
the space above the copied meta is not close to exhausted, which is the
whole case for searching rather than copying. (Costs in the bound are
deliberately UNDER-estimated, which only raises a roof; a roof may be
high, never low.)

**The bound self-audits.** My first pass at it used an approximated glut
branch and allocated 588 melons; the engine's own curve cut that to 161,
which lines up with the glut behaviour georgymamarin measured in
[the same thread](https://www.kaggle.com/competitions/kaggriculture/discussion/734412).
When a derivation and an independent measurement point the same way, the
number is probably real. The mix itself is led by strawberry (436 units,
\\$95k of gross): its scarcity amplitude is large and its demand median
is second only to wheat's.

**Tomato and egg belong in a schedule at MEDIAN demand.** The optimal
mix includes 218 tomato and the full egg cap of 108 with no scarcity
spike assumed. Since the 1.32.7 rebalance gave carrot, tomato and egg a
convex scarcity branch (the knee dots in the left figure; measured
ambient in the
[X-ray, section 6](https://www.kaggle.com/code/destbreso/a-week-four-x-ray-of-the-top-twelve)),
these goods also carry a fat right tail ON TOP of a mix they already
justify. The consensus routes barely touch them. That alone is a
schedule-shaped edge sitting in public view.

<a id="s4"></a>
## 4. Two legal representations, one law

The first design decision is WHAT the search mutates, and one engine
law governs it. Market orders settle by list index, one unit at a time,
and HIRE and BUY_LAND settle atomically at theirs. A recorded schedule
exploits this: its sells sit at lower indices than the purchases they
finance, on the same turn: the money that buys the day-12 land ARRIVES
two orders earlier in the same list. Splice the head of one stream onto
the tail of another and the financing chain breaks silently: a purchase
lands on an empty pocket, fails without an error, and everything
downstream executes against a farm that no longer matches the plan.
Every representation must keep that chain intact under every mutation
and every crossover.

Two representations satisfy the law, and they trade off against one
measured fact: a route's recorded CHOREOGRAPHY (who walks where, when)
carries about three quarters of its value. The strongest public
recorded stream banks 147.3k replayed verbatim against an idle
opponent; the same construction plan re-executed by a freshly derived
dispatcher recovers roughly half of that. Walking is the money.

* **Spec + compiler.** The genome is a declarative spec (quadrant
  unlock days, hire curve, herd waves, crop tables, a sell policy) and
  a COMPILER derives per-day targets plus the whole market channel from
  scratch. Coherent and financeable BY CONSTRUCTION under any mutation
  or block crossover. The right representation when you own an executor
  good enough to make walking cheap: the compiler is section 5.
* **Deltas over one stream.** The genome is a recorded stream plus a
  list of market-channel EDITS (shift a sell, split it, defer a good
  window, adjust a day of hires); the choreography is replayed verbatim
  and never touched. Financing-coherent because every edit is local to
  the one stream whose chain it inherits, and the zero-delta genome IS
  the naked stream: the search starts from the frontier instead of
  below it. The right representation when the choreography carries the
  value, which is the measured general case.

What stays illegal in both: splicing streams, and any mutation that
moves a sell without moving what it finances.

In [4]:
# the genome, in full: this is the entire search space of one candidate
GENOME_EXAMPLE = {
  "ne": 6, "sw": 10, "se": None,       # quadrant unlock days (None = skip)
  "plateau": 12, "ramp_full": 8,       # hands/day cap, days to reach it
  "tomato_day": 8, "melon2": 1,        # tomato window start; melon replant
  "sellpol": "hybrid",                 # daily | sweep | hybrid (section 5)
  "herd": [[0, "COW", 2], [0, "SHEEP", 2], [6, "COW", 2],
           [8, "SHEEP", 1], [10, "COW", 1]],          # [day, kind, n] waves
  "prog": {                            # tiles per crop, per quadrant
    "NW": {"WHEAT": 8, "CARROT": 3, "STRAWBERRY": 4, "MELON": 2},
    "NE": {"WHEAT": 10, "CARROT": 4, "STRAWBERRY": 5, "MELON": 6},
    "SW": {"WHEAT": 12, "CARROT": 5, "MELON": 8},
    "SE": {}}}

# macro-mutations: every move keeps the spec meaningful
MUTATIONS = [
  "shift a quadrant unlock day by 1 (or toggle SE on/off)",
  "plateau +-1, ramp +-2",
  "herd wave: n +-1, day +-2, or add a 1-animal wave",
  "move 1-2 tiles between crops within a quadrant (FALLOW allowed)",
  "tomato window +-2 days",
  "flip melon second wave",
  "switch sell policy"]

# block crossover: whole subsystems travel together, because crops x
# labour x cash are co-adapted through the shared hands and the shared
# pocket; uniform gene mixing would break exactly those linkages the
# building-block view of a GA wants preserved [1][2]
CROSSOVER_BLOCKS = ["unlock days", "hire curve", "herd waves",
                    "each quadrant's crop table", "tomato+melon", "sell policy"]
print("genome fields:", sorted(GENOME_EXAMPLE))
print("mutation operators:", len(MUTATIONS), "| crossover blocks:", len(CROSSOVER_BLOCKS))

<a id="s5"></a>
## 5. The compiler is the real algorithm

The GA on top is textbook. The part that decides whether the search can
work at all is the compiler, because it owns the gap between "a spec"
and "a schedule the engine will actually finance". Mine derives five
subsystems, in this order:

1. **Tile realisation.** Structures (pastures, coops) take the
   nearest-shed tiles of quadrants unlocked by their wave's day; crops
   fill each quadrant's remainder in a fixed priority (high-maintenance
   ongoing crops near the shed, melon far), clamped to the budget.
2. **Plant-day unrolling.** Cyclers (wheat, carrot) replant every
   maturity+1 days until the last harvest fits the horizon; melon gets
   one or two waves; ongoing crops (tomato, strawberry) plant once
   inside their window. This produces per-day construction targets and
   exact per-day seed requirements.
3. **A production projector.** Engine-exact cycle timing per crop, and
   for animals the fact of section 5.1 below. Its output is a cumulative
   sellable-by-morning series per good.
4. **The market channel.** Per day: sells at hour 0 (sized by the sell
   policy), purchases at hour 1 with seeds strictly before land, hires
   filling the remaining order slots (10 per turn).
5. **Financing by optimism.** Purchases are emitted at the spec's own
   days, not at days a cash model approves. This one is a finding, not a
   preference; it leads subsection 5.1.

The whole contract, as pseudocode. Every rule stated here is
engine-exact; what stays home is one implementation of it (section 8):

In [5]:
def compile_spec(spec):
    """spec -> {"days": {d: targets}, "market": {turn: orders}}."""

    # 1) TILE REALISATION
    #    each herd wave (day b, kind, n) takes the n nearest-shed tiles
    #    among quadrants unlocked by day b -> structure tiles (COOP/PASTURE);
    #    crops fill each quadrant's remainder from its table, ongoing crops
    #    nearest the shed, melon farthest, clamped to the 25-tile budget

    # 2) PLANT-DAY UNROLLING (per tile, engine crop table)
    #    cyclers : plant days a, a+m+1, a+2(m+1), ... while day+m <= 29
    #              (m = max_yield_day: wheat 4, carrot 3)
    #    melon   : one wave at a; optional second at a+13 if it ripens
    #    ongoing : one plant inside its window (tomato from the tomato_day
    #              gene, strawberry only if its quadrant opens by ~day 12)
    #    -> per-day construction targets + exact per-day seed needs

    # 3) PRODUCTION PROJECTOR -> cumulative sellable-by-morning per good
    #    crops   : yield x efficiency lands the morning after harvest
    #    animals : yield ACCRUES from placement at 1/interval per day but
    #              is harvestable only from first_yield_day, so the first
    #              harvest is a lump of about fyd/interval units (5.1)

    # 4) MARKET CHANNEL, per day d, under the 10-order cap per turn
    #    hour 0 : SELL each good, sized by the sell-policy gene:
    #             daily  = today's projected arrival x 1.25
    #             sweep  = cumulative projection minus already scheduled
    #             hybrid = daily pace + a cumulative top-up every 3rd day
    #             (near-exact on purpose: phantom demand re-steers, 5.1)
    #    hour 1 : BUY_SEED for due stages (seeds strictly BEFORE land),
    #             BUY_PRODUCT wheat for the herd's feed,
    #             BUY_ANIMAL waves, BUY_LAND on unlock days
    #    hours 1-2 : the day's HIREs, curve capped by farm size (~5
    #             work-tiles per hand) and NEVER cut by a cash forecast
    #    days 28-29 : terminal flush of every good, evening turns included

    # 5) FINANCING BY OPTIMISM
    #    every order at the spec's own day; the engine's silent refusals
    #    are the only brake, plus a thin repair layer re-issuing land and
    #    animal purchases while their window is still open (5.1)
    return {"days": "...", "market": "..."}

print("compiler contract: 5 derivations, every stated rule engine-exact")

<a id="s51"></a>
### 5.1 Four engine asymmetries that punish a cautious scheduler

**Failed orders are free; missing orders kill.** Every purchase in this
engine fails SILENTLY when the pocket is short: a HIRE that cannot be
paid simply does not happen, at zero cost. The converse is lethal: a
plant left unwatered two days running dies the second night, so a day
your schedule under-staffs out of caution is a day the engine starts
that countdown on every missed tile. The correct posture is asymmetric:
emit optimistically, let the engine's own refusals be the brake, and
keep a small repair layer that re-issues the purchases which matter
structurally (land, animals) while their window is still open.

**Selling is drainage, not revenue timing.** The shed caps at 100 units
and every deposit path into it respects the cap. A schedule that
under-sells for a few days does not merely defer revenue: its harvests
pile up in its workers' hands with nowhere to land. Daily sells are part
of the farm's plumbing.

**Animal yield arrives as a lump.** Yield accrues from PLACEMENT but
only becomes harvestable at the species' first-yield day, so the first
harvest of a cow pays out several units at once. A projector that
models a drip under-sells the lump, and the lump then sits against the
shed cap.

**Sells are demand signals, not just liquidations.** SELL settles per
unit and stops when the shed is empty, so slightly generous sells
self-clamp harmlessly. But if your executor harvests to order (mine
does: goods the schedule sells soon get harvested first), then a wildly
inflated sell is a phantom demand that re-steers the whole farm's
priorities. Sell sizing wants to be NEAR-exact with a self-healing
ledger, which is why the sell policy is a gene rather than a constant:
a daily-rate policy, a cumulative-sweep policy, and a hybrid differ by
double-digit percentages of final bank on the same spec (section 7).

### 5.2 The hire ledger, from the engine's own constants

The n-th hand hired in a day costs fib(n) starting 1, 1, 2, 3, 5. The
curve prices staffing decisions for you, and it is worth internalising:

In [6]:
def fib_cost(n_already):
    a, b = 1, 1
    for _ in range(n_already):
        a, b = b, a + b
    return a

daily = [sum(fib_cost(k) for k in range(n)) for n in range(16)]
print(f"12 hands cost ${daily[12]:,} a day; the 13th alone costs "
      f"${fib_cost(12):,}, the 14th ${fib_cost(13):,}.")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5, 3))
ns = list(range(1, 15))
ax.bar(ns, [daily[n] for n in ns], color="#2f6fb2", width=0.62)
for n in (8, 12, 14):
    ax.annotate(f"${daily[n]:,}", (n, daily[n]), xytext=(0, 4),
                textcoords="offset points", ha="center", fontsize=8.5,
                color="#333")
ax.set_xticks(ns)
ax.set_xlabel("hands hired in one day")
ax.set_ylabel("total cost of the day's hires, $")
ax.set_title("the Fibonacci hire ledger: the engine tells you\nwhere the plateau belongs", fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

<a id="s6"></a>
## 6. Why a genetic algorithm, and why islands

Simulated annealing [4] was my default, and it is a fine one: macro-moves
over a spec, temperature, done. Two properties of THIS problem argue for
a population instead.

**Recombination is legal and valuable here.** Crossover is only worth
having when recombining two good solutions plausibly yields a third, and
that requires the representation to keep co-adapted parts together. The
compiler guarantees financial coherence for any spec, and block
crossover at the subsystem level (a whole quadrant's crop table, the
whole herd plan) preserves the linkages that matter [1][2]. A good crop
mix and a good herd plan found by different lineages CAN combine. At the
stream level none of this holds, which is why annealing is the only
honest choice there, and why the compiler changes which algorithm is
available.

**Evaluation is expensive and parallel.** A candidate costs seconds of
real engine time, and an annealing chain is inherently sequential. A
population evaluates in parallel across every core. Islands [3] add the
classic benefits on top: independent basins explored concurrently,
diversity preserved by isolation, occasional ring migration to share
discoveries. With common random numbers (every candidate faces the same
fixed seed panel [6]) every comparison inside the search is paired, and
the eventual winner is re-read on a DISJOINT seed panel before anything
is claimed, the same screen/confirm discipline I argued in
[Six checks before you waste a submission](https://www.kaggle.com/code/destbreso/six-checks-before-you-waste-a-submission).
A miniature run of the public repo makes the point by itself: searched
on ONE seed for 75 generations, a genome gained 11.3k on its screening
seed and gave essentially all of it back on the disjoint panel, an
11.5k winner's-curse gap that the tool prints for you at the end.

And islands buy a measurement for free. If independent islands converge
to one design, the landscape says the optimal schedule is FORCED; if
they hold distinct niches at similar fitness, the landscape says a
PORTFOLIO of schedules exists (one per shop regime, say). Either answer
is worth having and a single population cannot give it.

<a id="s61"></a>
### 6.1 Why the islands are NOT seeded with known schedules

The tempting seeding is obvious: transcribe the public consensus route
and the strongest deduced schedules into specs, start every island from
proven material. I decided against it, and the intuitive reason
(ownership: a search seeded with someone's route produces a derivative
of it) is the weakest of the three.

**Selection takeover.** A transcribed strong schedule enters the
population as a super-individual. Under tournament selection its
descendants flood a small island in a handful of generations, and ring
migration then exports the takeover everywhere; expected takeover time
under tournament selection grows only logarithmically with population
size [7]. The simulation below makes it concrete: even a 10 % fitness
edge owns half a 40-slot population by about generation four. (The 2 %
curve is the flip side: an edge the evaluation noise can hide does not
spread at all, which is exactly why the search pairs every comparison
on common random numbers.) After a takeover the search is not exploring
the schedule space, it is polishing one point of it. You have rebuilt the copying road with extra steps and
spent a GA to do it.

**Transcription loss makes the head start illusory.** A recorded route
is co-adapted to ITS author's execution layer: its plant days assume its
walking efficiency, its financing assumes its harvest cadence. A spec
captures counts and days, not positional choreography, so the
transcription is NOT the strong agent, it is a mediocre shadow of one.
Seeding with it gets the worst of both worlds: enough fitness to take
over the population, none of the quality that made the original good.
I measured this co-adaptation directly when
[grafting between recorded routes](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay):
a route's construction is a positional choreography, and it does not
survive being moved onto a different executor.

**It destroys the experiment.** The convergence-versus-niches
measurement of the previous section only means something if the islands
start independent. Seed them all from one meta and you have answered
the question by assumption.

So the four species are grown from PRIORS, not from streams: one from
the leader envelope statistics (the quadrant windows, hire plateaus and
crop ranges measured in the X-ray, which are aggregates, not anyone's
route), one from the relaxed bound's optimal mix (tomato, egg, geese),
one high-intensity variant (fourth quadrant, maximum plateau), one
random-within-envelope. Known schedules keep exactly one job: the
BASELINE the search must beat.

In [7]:
import numpy as np

rng = np.random.default_rng(7)
N, K, GENS, RUNS = 40, 3, 12, 400

def takeover(adv):
    frac = np.zeros((RUNS, GENS + 1))
    for r in range(RUNS):
        fit = np.full(N, 1.0) + rng.normal(0, 0.02, N)
        seed = np.zeros(N, bool)
        seed[0], fit[0] = True, adv
        frac[r, 0] = seed.mean()
        for g in range(1, GENS + 1):
            idx = rng.integers(0, N, (N, K))
            win = idx[np.arange(N), np.argmax(fit[idx], axis=1)]
            fit, seed = fit[win] + rng.normal(0, 0.005, N), seed[win]
            frac[r, g] = seed.mean()
    return frac.mean(0)

import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
for adv, col, lab in ((1.02, "#2f6fb2", "2 % fitter: hidden by noise, does not spread"),
                      (1.10, "#c2571f", "10 % fitter")):
    ax1.plot(range(GENS + 1), takeover(adv), "-o", ms=4, color=col,
             label=f"seeded schedule {lab}")
ax1.axhline(0.5, color="#bbb", lw=1, ls=":")
ax1.set_xlabel("generation")
ax1.set_ylabel("share of population descended\nfrom the seeded schedule")
ax1.set_title("tournament takeover (pop 40, k=3, 400 runs):\none good seed owns the search in a few generations", fontsize=10)
ax1.legend(frameon=False, fontsize=9, loc="lower right")
ax1.set_ylim(0, 1.02)
for s in ("top", "right"):
    ax1.spines[s].set_visible(False)

names = ["envelope", "bound\nmix", "intensity", "random"]
descr = ["leader statistics", "tomato, egg, geese", "4th quadrant, max hires",
         "within the envelope"]
cols = ["#2f6fb2", "#c2571f", "#7b5bb5", "#9aa3ad"]
pos = [(0, 1.05), (1.15, 0), (0, -1.05), (-1.15, 0)]
off = [(0, 0.62, "center", "bottom"), (0.58, 0, "left", "center"),
       (0, -0.62, "center", "top"), (-0.58, 0, "right", "center")]
for (x, y), (dx, dy, ha, va), name, dsc, col in zip(pos, off, names, descr,
                                                    cols):
    ax2.add_patch(plt.Circle((x, y), 0.44, color=col, alpha=0.88, zorder=3))
    ax2.annotate(name, (x, y), ha="center", va="center", fontsize=7.8,
                 color="white", zorder=4)
    ax2.annotate(dsc, (x + dx, y + dy), ha=ha, va=va, fontsize=8,
                 color="#777", zorder=4)
for i in range(4):
    x1, y1 = pos[i]
    x2, y2 = pos[(i + 1) % 4]
    ax2.annotate("", xy=(x2 * 0.58, y2 * 0.58), xytext=(x1 * 0.58, y1 * 0.58),
                 arrowprops=dict(arrowstyle="->", color="#888", lw=1.3,
                                 connectionstyle="arc3,rad=-0.3"))
ax2.text(0, -2.02, "best of island i replaces the worst of island i+1, "
         "every 6 generations", ha="center", fontsize=8.5, color="#555")
ax2.set_xlim(-2.6, 2.6); ax2.set_ylim(-2.25, 1.95)
ax2.set_aspect("equal"); ax2.axis("off")
ax2.set_title("four OWNED species on a migration ring", fontsize=10)
plt.tight_layout(); plt.show()

In [8]:
# the search loop, complete except for compile_spec() and the engine
# evaluator (see section 8 for what those are and why they stay home)
import copy, random

def run_search(species, evaluate, compile_spec, hours=3, seeds=(11, 23, 47)):
    rng = random.Random(20260824)
    islands = [[sp] + [mutate(sp, rng) for _ in range(9)] for sp in species]
    cache = {}                                    # gid -> {seed: bank}

    def fitness(g):
        r = cache.get(gid(g), {})
        return sum(r.values()) / len(r) if len(r) == len(seeds) else None

    for gen in range(10**9):                      # wall clock breaks the loop
        for pop in islands:                       # evaluate the fresh genomes
            for g in pop:                         # (parallel pool in the real
                for s in seeds:                   #  build; every candidate on
                    if s not in cache.setdefault(gid(g), {}):   # the SAME
                        cache[gid(g)][s] = evaluate(compile_spec(g), s)  # CRN
        nxt = []
        for pop in islands:
            pop.sort(key=lambda g: -fitness(g))
            child_pop = [pop[0]]                  # elitism: best survives
            while len(child_pop) < len(pop):
                a, b = (tournament(pop, rng, fitness) for _ in "ab")
                child = crossover(a, b, rng) if rng.random() < 0.6                     else copy.deepcopy(a)
                child_pop.append(mutate(child, rng))
            nxt.append(child_pop)
        islands = nxt
        if gen % 6 == 5:                          # ring migration: best of
            bests = [max(p, key=fitness) for p in islands]      # island i-1
            for i, pop in enumerate(islands):     # replaces worst of island i
                pop.sort(key=fitness)
                pop[0] = copy.deepcopy(bests[i - 1])
    # afterwards: re-read the winner on a DISJOINT confirm panel; the
    # screening mean of a selected best is inflated (winner's curse) [8]

def tournament(pop, rng, fitness, k=3):
    return max(rng.sample(pop, k), key=fitness)

print("search skeleton defined: species x islands x CRN x ring migration")

The evaluator is the one piece anyone can rebuild in an afternoon,
because the engine ships in `kaggle_environments`. The pattern that
matters is fixing the seed (it reproduces weeds and shop draws, so
comparisons pair) and reading the observation the same way the live
framework does. A minimal harness, runnable right here:

In [9]:
from kaggle_environments import make

PASS = {"farmer": ["PASS"], "hands": [], "market": []}

def bank_of(agent_fn, seed, steps=720):
    env = make("kaggriculture",
               configuration={"episodeSteps": steps, "seed": int(seed)})
    env.reset(2)
    while not env.done:
        shared = env.state[0].observation         # seat 1 gets a trimmed
        obs = dict(env.state[0].observation)      # observation; merge the
        for k, v in shared.items():               # shared fields like the
            obs.setdefault(k, v)                  # live framework does
        env.step([agent_fn(obs), dict(PASS)])
    return float(env.state[0].reward or 0)

idle_bank = bank_of(lambda obs: dict(PASS), seed=11)
print(f"an agent that does nothing banks ${idle_bank:,.0f} on seed 11:")
print("that is the zero of the whole corridor in section 3")

<a id="s7"></a>
## 7. The campaign: what the searches measured

Several searches have now run under this machinery, across both
representations of section 4. The numbers below are the ones that
changed how the NEXT search is built, which is the honest definition of
progress in a search campaign. First, two opening facts from the
spec+compiler representation, because they are method results, not
schedule results.

**The species really are diverse.** Four different priors produce four
different farms, spread across a 36-54k band on the screen panel before
the search moves a single gene. That spread is the raw material
selection needs.

**The sell policy gene matters more than intuition says.** The same
spec, compiled under the three sell policies, differs by 14k on one
seed: drainage details are worth more than most crop swaps. This is the
kind of fact a search surfaces and a hand-designer skips past.

In [10]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.4))

species = ["envelope", "intensity", "random", "bound-mix"]
banks = [53.7, 44.3, 41.3, 36.3]
cols = ["#2f6fb2", "#7b5bb5", "#9aa3ad", "#c2571f"]
b = ax1.barh(species[::-1], banks[::-1], color=cols[::-1], height=0.55)
for r, v in zip(b, banks[::-1]):
    ax1.annotate(f"{v:.1f}k", (v, r.get_y() + r.get_height() / 2),
                 xytext=(4, 0), textcoords="offset points",
                 va="center", fontsize=9, color="#333")
ax1.set_title("four owned species, seed 11, before any search", fontsize=10)
ax1.set_xlabel("bank vs idle, thousands of $")
ax1.set_xlim(0, 92)
ax1.axvline(83.4, color="#555", lw=1, ls="--")
ax1.annotate("consensus route\nreplayed: 83.4k", (83.4, 3.1), fontsize=8,
             ha="right", xytext=(-6, 0), textcoords="offset points", color="#555")
for s in ("top", "right"):
    ax1.spines[s].set_visible(False)

pols = ["daily", "sweep", "hybrid"]
pb = [39.6, 42.6, 53.7]
b2 = ax2.bar(pols, pb, color=["#9aa3ad", "#9aa3ad", "#1f9d6b"], width=0.5)
for r, v in zip(b2, pb):
    ax2.annotate(f"{v:.1f}k", (r.get_x() + r.get_width() / 2, v),
                 xytext=(0, 4), textcoords="offset points",
                 ha="center", fontsize=9, color="#333")
ax2.set_title("one spec, three sell policies, seed 11", fontsize=10)
ax2.set_ylabel("bank vs idle, thousands of $")
ax2.set_ylim(0, 62)
for s in ("top", "right"):
    ax2.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

**The finding that redesigned the objective.** A stream-delta
search (representation two of section 4, on the public frontier streams
of section 3) optimised the equal-weighted mean bank over three
opponents: an idle farm, the strongest fixed rival, and an adaptive
public rival with live clone machinery. Its best genome, 24 market
deltas on the strongest base, decomposed like this on a DISJOINT
confirm panel:

In [11]:
import matplotlib.pyplot as plt

cats = ["alone\n(vs idle)", "in company\n(vs rivals)"]
base = [136.8, 87.3]
winner = [149.9, 73.4]
x = [0, 1]
fig, ax = plt.subplots(figsize=(8.5, 3.6))
b1 = ax.bar([i - 0.18 for i in x], base, width=0.32, color="#9aa3ad",
            label="zero-delta base stream")
b2 = ax.bar([i + 0.18 for i in x], winner, width=0.32, color="#c2571f",
            label="searched genome (24 market deltas)")
for bars in (b1, b2):
    for r in bars:
        ax.annotate(f"{r.get_height():.1f}k",
                    (r.get_x() + r.get_width() / 2, r.get_height()),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=9)
ax.annotate("+13.2k", (0.52, 160), ha="center", fontsize=11,
            color="#14724d", weight="bold")
ax.annotate("-13.9k", (1.52, 96), ha="center", fontsize=11,
            color="#b5486b", weight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylim(0, 172)
ax.set_ylabel("mean bank, disjoint confirm panel\n(thousands of $)")
ax.set_title("the same 24 deltas, two opposite verdicts: alone they\n"
             "EARN, in company they EXPOSE", fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper right")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.show()

Both effects are real and out of sample: the deltas genuinely earn
+13.2k against an empty market and genuinely lose 13.9k against rivals,
because the sell micro-timing that squeezes empty prices exposes the
schedule exactly where rivals contest it. An equal-weight objective let
one axis buy the other one-for-one, which is how this genome screened
as a champion and confirmed as a regression. Two consequences now
govern the campaign:

* **The objective is priced in LADDER currency.** The competition
  scores wins, not banks (its final standings are a Bradley-Terry fit
  on W/L, and in a public all-vs-all I ran, mean banks compressed into
  a 3k band while win counts ranged from 6 to 54). So fitness is now
  Pr[win] = Phi(margin/sigma) per rival, company-weighted, with
  monetisation kept only as a small health term: smooth like a bank
  margin, priced like the leaderboard.
* **Screen panels got three orders of magnitude wider.** A tiny fixed
  screen is LEARNABLE over enough generations (measured: a 17.7 %
  screen-to-confirm gap after 94 of them), and the fix is the
  [bit-exact C++ engine](https://www.kaggle.com/code/destbreso/kagsim-the-engine-at-2000-episodes-per-second):
  the fixed-rival axes of every fitness call now run 32-seed margin
  panels in milliseconds, which also supplies the within-pairing sigma
  the Phi objective needs. The engine backend is a config parameter:
  `real` for full fidelity, `hybrid` for simulator-wide panels with the
  adaptive rival kept on the real engine.


### 7.1 The long-run failure mode, and its countermeasures

My longest run so far went 144 generations and 8,445 games, and it
taught two lessons the 25-generation pilot could not.

**Premature convergence wastes the tail of a long run.** Every island
ended up holding the same champion, and from at least generation 111 to
the end nothing changed: same four numbers, generation after
generation, evaluating almost nothing new (the cache sees no new
genomes). Thirty-plus generations of wall clock spent polishing one
point of the space.

**The screen panel gets LEARNED.** That run screened at 49.5k and
confirmed at 36.0k on the disjoint panel: a 13.5k winner's-curse gap,
double the pilot's 6.8k at 25 generations. More generations on the same
three seeds do not just stall, they overfit: the search finds genomes
that exploit those seeds' particular shop draws. A long run needs a
bigger `seeds` panel, and the confirm gap is the tell that yours was
too small.

The repo now carries countermeasures for the first problem, all config
parameters: a per-island STAGNATION counter (improving means beating
your own best, not holding it), island CATACLYSM (a flat island keeps
its elite and is reseeded around it with fresh species draws and hot
mutants), HYPERMUTATION (a half-stagnant island mutates 2-5 moves
instead of 1-3), an ARCHIPELAGO RESTART (global best flat too long:
only the champion's island survives), and random IMMIGRANTS every few
generations. Island count is a parameter too (`islands: 8` cycles the
species list).

Validated the direct way: a run with aggressively low thresholds, so
the mechanisms fire early and often. Every restart produced a new
record within two generations:

In [12]:
import matplotlib.pyplot as plt

GENS = list(range(1, 17))
BEST = [32095, 32095, 32095, 32095, 34250, 36721, 36721, 36721, 36721,
        36721, 38902, 38902, 43629, 43629, 43629, 45245]
RESTARTS = [4.5, 9.5]

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(GENS, [b / 1000 for b in BEST], "-o", ms=4, color="#2f6fb2",
        label="global best")
for i, x in enumerate(RESTARTS):
    ax.axvline(x, color="#c2571f", lw=1.2, ls="--")
    ax.annotate("island restart\n(flat 4 gens)", (x, 33.2 + 6 * i),
                ha="center", fontsize=8, color="#c2571f",
                xytext=(x + (1.6 if i == 0 else 1.8), 32.6 + 5.5 * i),
                textcoords="data",
                arrowprops=dict(arrowstyle="->", color="#c2571f", lw=1))
ax.set_xlabel("generation")
ax.set_ylabel("global best, bank vs idle\n(thousands of $)")
ax.set_title("escape validation (aggressive thresholds): every restart is\nfollowed by a new record within two generations", fontsize=10)
ax.set_xticks(GENS)
ax.legend(frameon=False, fontsize=9, loc="lower right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

<a id="s8"></a>
## 8. What stays home, and the honest reason why

The pipeline itself is public: the
[companion repo](https://github.com/destbreso/kaggriculture-island-ga)
carries the compiler, the island GA, the CRN evaluation and a CLI that
ends in screen/confirm metrics, plus a deliberately SIMPLE reference
executor so everything runs end to end out of the box.

Two things are not in it. My EXECUTION layer: the dispatcher that turns
targets into per-turn verbs is where a season's worth of my measurement
lives, and it is the multiplier on any schedule. The repo makes that
claim measurable instead of rhetorical: the same genome on the same
seed banks 27k under the reference executor and 54k under mine. And the
SCHEDULES my own runs return, for the reason section 2 already made: a
fielded schedule is deducible from replays within days (my own DNA
tooling proves it), so its value is highest exactly while it is fresh,
and this competition is still running.

What I hope the method buys you is the same thing it bought me: a base
that is yours, priced against a bound you computed, grown by a search
you can re-run when the meta moves. The layer on top is a different
story, and the field's own evidence says the base is where the cheap
points are.

If you build on this, the pieces I would genuinely love to see compared:
your corridor numbers, whether your islands converge or hold niches,
and whether your landscape also ranks drainage genes above crop genes.

<a id="s9"></a>
## References

Technical:

1. J. H. Holland, *Adaptation in Natural and Artificial Systems*,
   University of Michigan Press, 1975.
2. D. E. Goldberg, *Genetic Algorithms in Search, Optimization and
   Machine Learning*, Addison-Wesley, 1989.
3. D. Whitley, S. Rana, R. B. Heckendorn, "The Island Model Genetic
   Algorithm: On Separability, Population Size and Convergence",
   *Journal of Computing and Information Technology*, 1999.
4. S. Kirkpatrick, C. D. Gelatt, M. P. Vecchi, "Optimization by
   Simulated Annealing", *Science* 220, 1983.
5. M. R. Garey, D. S. Johnson, *Computers and Intractability: A Guide
   to the Theory of NP-Completeness*, W. H. Freeman, 1979.
6. A. M. Law, W. D. Kelton, *Simulation Modeling and Analysis*,
   McGraw-Hill (common random numbers as variance reduction).
7. D. E. Goldberg, K. Deb, "A Comparative Analysis of Selection Schemes
   Used in Genetic Algorithms", *Foundations of Genetic Algorithms*,
   1991.
8. D. D. Jensen, P. R. Cohen, "Multiple Comparisons in Induction
   Algorithms", *Machine Learning* 38, 2000.

Community work this builds on:

* Rayk Kretzschmar (@raykkretzschmar),
  [Kaggriculture Rank Your Agent](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent):
  the public consensus construction used as the replay baseline.
* georgymamarin (@georgymamarin),
  [Kaggriculture Episodes](https://www.kaggle.com/datasets/georgymamarin/kaggriculture-episodes)
  and the melon and glut measurements in
  [discussion 734412](https://www.kaggle.com/competitions/kaggriculture/discussion/734412).
* Luka Duvanov (@nekkon),
  [The top agent is a 720-turn replay, not a strategy](https://www.kaggle.com/code/nekkon/the-top-agent-is-a-720-turn-replay-not-a-strategy).
* The measurement engine the campaign now runs on:
  [kagsim, the bit-exact C++ engine](https://www.kaggle.com/code/destbreso/kagsim-the-engine-at-2000-episodes-per-second)
  and its repo
  [kaggriculture-cppsim](https://github.com/destbreso/kaggriculture-cppsim).
* The runnable companion code:
  [github.com/destbreso/kaggriculture-island-ga](https://github.com/destbreso/kaggriculture-island-ga)
  (MIT).
* My own measurement series is linked inline throughout.